# Data Cleaning 09 -- WRDS Macro Daily

## Input
`Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet` (5,485 rows, 2004-01-02 to 2024-12-31, keyed on date only)

## Purpose
Cleans daily macro/market-level data from WRDS. Contains VIX family OHLC (S&P 500, S&P 100, Nasdaq-100, DJIA), Fama-French 5 factors + momentum + risk-free rate, 11 FX rates, and 12 world index returns. Key concerns addressed: placeholder value detection, weekend artefact rows, discontinued series (VXO), holiday-driven NaN gaps, and verification that extreme values are genuine market events rather than data errors.

## Stage 0: Load & Inspect
Basic shape, date range, column inventory, and dtype verification.

## Stage 1: Missing Data Audit
- Total NaN count and percentage across all factor cells
- Per-column NaN counts with first/last valid dates and flags
- Per-row NaN distribution

## Stage 2: Placeholder Value Detection
- Checks all factor columns for common placeholder values (-99, -999, -9999, 99, 999, 9999, -99.99, -999.99, etc.)
- Identifies columns with suspiciously common exact values (>20% of observations identical)
- Flags columns with excessive zeros (>30%)
- Identifies potential outlier placeholders (values >10 standard deviations from mean)

## Stage 3: Date Structure & Gap Analysis
- **Day-of-week distribution:** identifies weekend rows
- **Weekend row inspection:** detailed investigation of 10 weekend rows (6 Saturdays, 4 Sundays), showing which columns have valid values and confirming the following Monday has full data
- **Date gap distribution:** reports gap statistics and identifies long gaps (>5 days) with specific dates (e.g., Hurricane Sandy Oct 2012, holiday stretches)
- **Duplicate date check**

## Stage 4: Per-Factor Gap Analysis
For each factor, identifies the longest consecutive NaN run with start/end dates. Confirms all gaps are holiday-driven (VIX/FF max 2 days, FX max 2 days, world indices max 3--8 days from country-specific holidays like Chinese New Year and Japanese Golden Week).

## Stage 5: Value Range & Quality Checks
- Full summary statistics for all factor columns (min, median, max, mean, std)
- Scale check: identifies columns in decimal scale vs percentage scale
- Constant or near-constant column detection

## Stage 7: Clean & Save

### Columns Dropped (4) -- VXO Discontinued
- `vxo`, `vxoo`, `vxoh`, `vxol`: CBOE's old S&P 100 Volatility Index. Discontinued January 2021 -- 1,041 consecutive NaN days from 2021-01-08 to 2024-12-31 (21.9% NaN overall). VIX is the modern replacement and has full coverage.

### Rows Dropped (10) -- Weekend Artefacts
6 Saturdays and 4 Sundays with at most 1 valid value out of 46 columns. The valid values are stray noise: `widx_ind` microreturns (~2e-06) from Indian market special Saturday sessions, and a single `vxd` observation. Every following Monday has 40--46 valid columns. Keeping these would pollute forward-fill (Friday to Saturday instead of Friday to Monday).

### No Placeholder Contamination Found
- `fx_jpy = 99` and `fx_krw = 999` are legitimate exchange rates within normal ranges (JPY: 75--162, KRW: 903--1,570)
- >10-sigma outliers are real market events: `mktrf = -0.1201` (March 2020 COVID crash), `umd = -0.1437` (momentum crash), `widx_ind = +0.159` (Indian market rally). No action needed.

### `rf` Retained Despite Only 3 Unique Values
The daily risk-free rate was 0.0000 during ZIRP (2008--2015) and 0.0001--0.0002 otherwise. Not useful as a standalone factor, but the merge pipeline needs it for computing excess returns (return - rf).

### No Winsorisation
Applied in the merge pipeline.

### No Forward-Fill Applied Here
Remaining NaN in all factors is holiday-driven with short runs (max 2--8 days depending on the series). Forward-fill happens in the merge pipeline when aligning to trading days.

## Output
`Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_daily_clean.parquet` -- 42 factor columns (down from 46)

In [1]:
# %% [markdown]
# # Data Cleaning: macro_daily.parquet
#
# Source: Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet
# Output: Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_daily_clean.parquet
#
# Daily macro/market-level data from WRDS. This is keyed on date only (no PERMNO).
# Likely contains Fama-French factors, market indices, VIX, Treasury yields,
# FX rates, and other market-wide variables.
#
# Key concerns:
#   - Macro data: forward-fill IS appropriate (single time series)
#   - Check for placeholder values (-99, 99, -999, 0) masking real NaN
#   - Weekend/holiday handling
#   - Gap analysis for each series

# %%
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path('../../../Data/Data_Collection/Initial/09_Macro_Daily_Monthly_WRDS/macro_daily.parquet')
OUT_DIR  = Path('../../../Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 0: LOAD & INSPECT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 0: LOAD & INSPECT — Macro Daily")
print("=" * 90)

df = pd.read_parquet(RAW_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

factor_cols = [c for c in df.columns if c != 'date']

print(f"\n  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"  Unique dates: {df['date'].nunique():,}")
print(f"  Factor columns: {len(factor_cols)}")

print(f"\nColumns and dtypes ({len(factor_cols)} factors):")
for i, c in enumerate(factor_cols, 1):
    print(f"  {i:>3d}. {c:<35s} {str(df[c].dtype):<15s}")

print(f"\n--- Head (10 rows) ---")
show_cols = ['date'] + factor_cols[:8]
print(df[show_cols].head(10).to_string(index=False))

print(f"\n--- Tail (10 rows) ---")
print(df[show_cols].tail(10).to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 1: MISSING DATA AUDIT
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 1: MISSING DATA AUDIT")
print("=" * 90)

n_rows = len(df)

# ── Total NaN ────────────────────────────────────────────────────────────────
total_cells = n_rows * len(factor_cols)
total_nan = df[factor_cols].isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Total NaN:   {total_nan:,} ({total_nan / total_cells * 100:.2f}%)")

# ── Per-column NaN with first/last valid dates ───────────────────────────────
col_nan = df[factor_cols].isna().sum()
col_nan_pct = (col_nan / n_rows * 100).round(2)
col_nan_sorted = col_nan_pct.sort_values(ascending=False)

print(f"\n--- Per-Column NaN ---")
print(f"\n  {'Column':<35s} {'NaN %':>8s}  {'Count':>8s}  {'First Valid':>12s}  {'Last Valid':>12s}")
print("  " + "-" * 80)
for col, pct in col_nan_sorted.items():
    count = int(col_nan[col])
    valid = df[df[col].notna()]['date']
    first = valid.min().date() if len(valid) > 0 else 'N/A'
    last = valid.max().date() if len(valid) > 0 else 'N/A'
    flag = " ← DROP" if pct >= 30 else (" ← INVESTIGATE" if pct >= 10 else "")
    print(f"  {col:<35s} {pct:>7.2f}%  {count:>8,d}  {str(first):>12s}  {str(last):>12s}{flag}")

# ── Per-row NaN ──────────────────────────────────────────────────────────────
row_nan = df[factor_cols].isna().sum(axis=1)
print(f"\n--- Per-Row NaN Distribution ---")
print(f"  Rows with 0 NaN: {(row_nan == 0).sum():>8,d} ({(row_nan == 0).mean()*100:.1f}%)")
print(f"  Rows with 1-5 NaN: {((row_nan >= 1) & (row_nan <= 5)).sum():>8,d}")
print(f"  Rows with 6-15 NaN: {((row_nan > 5) & (row_nan <= 15)).sum():>8,d}")
print(f"  Rows with >15 NaN: {(row_nan > 15).sum():>8,d}")
print(f"  Max NaN in any row: {row_nan.max()} out of {len(factor_cols)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 2: PLACEHOLDER VALUE DETECTION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 2: PLACEHOLDER VALUE DETECTION")
print("=" * 90)

# Common placeholders: -99, -999, 99, 999, -99.99, 0 (when unexpected)
placeholders = [-99, -999, -9999, 99, 999, 9999, -99.99, -999.99, 99.99, 999.99]

print(f"\n--- Checking for common placeholder values ---")
for val in placeholders:
    hits = {}
    for col in factor_cols:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        n = (df[col] == val).sum()
        if n > 0:
            hits[col] = n
    if hits:
        print(f"\n  Value = {val}:")
        for col, n in sorted(hits.items(), key=lambda x: -x[1]):
            pct = n / len(df[col].dropna()) * 100
            print(f"    {col:<35s} {n:>6d} occurrences ({pct:.2f}%)")

# ── Check for suspiciously repeated exact values ─────────────────────────────
print(f"\n--- Columns with suspiciously common exact values ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    top_val = vals.value_counts().iloc[0]
    top_pct = top_val / len(vals) * 100
    if top_pct > 20:  # more than 20% the same value
        most_common = vals.value_counts().index[0]
        print(f"  {col:<35s} value {most_common:>10.4f} appears {top_val:,} times ({top_pct:.1f}%)")

# ── Check for columns that are all zeros ─────────────────────────────────────
print(f"\n--- Columns with excessive zeros ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    n_zero = (vals == 0).sum()
    pct_zero = n_zero / len(vals) * 100
    if pct_zero > 30:
        print(f"  {col:<35s} {n_zero:,} zeros ({pct_zero:.1f}%)")

# ── Check for extreme outliers that might be placeholders ────────────────────
print(f"\n--- Potential outlier placeholders (values >10 std from mean) ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) < 100:
        continue
    mean = vals.mean()
    std = vals.std()
    if std == 0:
        continue
    n_extreme = (((vals - mean).abs() / std) > 10).sum()
    if n_extreme > 0:
        extremes = vals[((vals - mean).abs() / std) > 10]
        print(f"  {col:<35s} {n_extreme:>4d} values >10σ  "
              f"(range: [{extremes.min():.4f}, {extremes.max():.4f}], "
              f"mean: {mean:.4f}, std: {std:.4f})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 3: DATE STRUCTURE & GAP ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 3: DATE STRUCTURE & GAP ANALYSIS")
print("=" * 90)

# ── Day-of-week distribution ────────────────────────────────────────────────
print(f"\n--- Day-of-week distribution ---")
dow = df['date'].dt.day_name().value_counts()
print(dow.to_string())

has_weekends = (df['date'].dt.dayofweek >= 5).any()
print(f"\n  Contains weekends: {'YES' if has_weekends else 'NO'}")

# ── Date gap analysis ───────────────────────────────────────────────────────
print(f"\n--- Date gap distribution ---")
date_diffs = df['date'].diff().dt.days.dropna()
print(f"  Mean gap: {date_diffs.mean():.1f} days")
print(f"  Median gap: {date_diffs.median():.0f} days")
print(f"  Min gap: {date_diffs.min():.0f} days")
print(f"  Max gap: {date_diffs.max():.0f} days")
print(f"\n  Gap distribution:")
for gap, count in date_diffs.value_counts().sort_index().head(10).items():
    print(f"    {int(gap):>3d} days: {count:>5,d}")

# ── Long gaps (>5 days) ─────────────────────────────────────────────────────
long_gaps = date_diffs[date_diffs > 5]
if len(long_gaps) > 0:
    print(f"\n  Gaps > 5 days: {len(long_gaps)}")
    for idx in long_gaps.sort_values(ascending=False).head(15).index:
        gap_end = df.loc[idx, 'date']
        gap_start = df.loc[idx - 1, 'date'] if idx > 0 else None
        if gap_start:
            print(f"    {gap_start.date()} → {gap_end.date()} ({int(date_diffs.loc[idx])} days)")
else:
    print(f"\n  ✓ No gaps > 5 days")

# ── Duplicate dates ──────────────────────────────────────────────────────────
print(f"\n--- Duplicate dates ---")
n_dupes = df['date'].duplicated().sum()
if n_dupes == 0:
    print(f"  ✓ No duplicate dates")
else:
    print(f"  ⚠ {n_dupes} duplicate dates")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 4: PER-FACTOR GAP ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 4: PER-FACTOR GAP ANALYSIS")
print("=" * 90)

# For each factor, find the longest consecutive NaN run
print(f"\n--- Longest consecutive NaN runs per factor ---")
print(f"\n  {'Factor':<35s} {'Max Run':>8s}  {'Total NaN':>10s}  {'NaN %':>7s}  {'Where'}")
print("  " + "-" * 85)

for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    is_nan = df[col].isna()
    total_nan_col = is_nan.sum()
    if total_nan_col == 0:
        continue

    # Find longest run
    runs = is_nan.ne(is_nan.shift()).cumsum()
    nan_runs = is_nan.groupby(runs).sum()
    nan_runs = nan_runs[nan_runs > 0]

    if len(nan_runs) == 0:
        continue

    max_run = int(nan_runs.max())
    max_run_idx = nan_runs.idxmax()

    # Find date of longest run
    run_rows = df[runs == max_run_idx]
    run_start = run_rows['date'].iloc[0].date()
    run_end = run_rows['date'].iloc[-1].date()

    pct = total_nan_col / n_rows * 100
    print(f"  {col:<35s} {max_run:>8d}  {total_nan_col:>10,d}  {pct:>6.2f}%  {run_start} → {run_end}")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 5: VALUE RANGE & QUALITY CHECKS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 5: VALUE RANGE & QUALITY CHECKS")
print("=" * 90)

print(f"\n--- Summary statistics ---")
print(f"\n  {'Column':<35s} {'min':>12s}  {'median':>12s}  {'max':>12s}  {'mean':>12s}  {'std':>12s}")
print("  " + "-" * 100)
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:<35s} (non-numeric)")
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        print(f"  {col:<35s} (all NaN)")
        continue
    print(f"  {col:<35s} {vals.min():>12.4f}  {vals.median():>12.4f}  "
          f"{vals.max():>12.4f}  {vals.mean():>12.4f}  {vals.std():>12.4f}")

# ── Check for columns that might be percentage vs decimal ────────────────────
print(f"\n--- Scale check: percentage vs decimal ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    if vals.min() >= -1 and vals.max() <= 1 and 'return' not in col.lower():
        print(f"  {col:<35s} range [{vals.min():.6f}, {vals.max():.6f}] — decimal scale")
    elif vals.min() >= -100 and vals.max() <= 100 and vals.std() > 0.5:
        print(f"  {col:<35s} range [{vals.min():.4f}, {vals.max():.4f}] — possibly percentage")

# ── Constant or near-constant columns ────────────────────────────────────────
print(f"\n--- Constant or near-constant columns ---")
for col in factor_cols:
    if not pd.api.types.is_numeric_dtype(df[col]):
        continue
    vals = df[col].dropna()
    if len(vals) == 0:
        continue
    if vals.nunique() <= 3:
        print(f"  {col:<35s} only {vals.nunique()} unique values: {sorted(vals.unique()[:5])}")
    elif abs(vals.mean()) > 1e-10 and vals.std() / abs(vals.mean()) < 0.001:
        print(f"  {col:<35s} near-constant (cv = {vals.std()/abs(vals.mean()):.6f})")

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 6: SUMMARY — DECISIONS NEEDED
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STAGE 6: SUMMARY — DECISIONS NEEDED")
print("=" * 90)

print(f"""
Review the output above:

1. COLUMNS TO DROP:
   - Any column with ≥30% NaN
   - Constant or near-constant columns
   - Columns with discontinued data (long tail NaN gap to present)
   - Placeholder-contaminated columns (if not fixable)

2. PLACEHOLDER VALUES:
   - Replace identified placeholders with NaN before any forward-fill
   - Common in WRDS: -99, -999, -99.99

3. DATE RANGE:
   - Trim to 2004-01-01 if earlier data exists
   - Drop weekends if present (macro daily should be trading days only)

4. NaN HANDLING:
   - This is macro data — forward-fill IS appropriate
   - But only after cleaning placeholders
   - Forward-fill in merge pipeline, not here
   - Leave dates and NaN as-is in cleaned file

Paste back the output and I will write the cleaning cell.
""")

STAGE 0: LOAD & INSPECT — Macro Daily

  Shape: 5,485 rows × 47 columns
  Date range: 2004-01-02 → 2024-12-31
  Unique dates: 5,485
  Factor columns: 46

Columns and dtypes (46 factors):
    1. vix                                 Float64        
    2. vixo                                Float64        
    3. vixh                                Float64        
    4. vixl                                Float64        
    5. vxo                                 Float64        
    6. vxoo                                Float64        
    7. vxoh                                Float64        
    8. vxol                                Float64        
    9. vxn                                 Float64        
   10. vxno                                Float64        
   11. vxnh                                Float64        
   12. vxnl                                Float64        
   13. vxd                                 Float64        
   14. vxdo                                Flo

In [2]:
# %%
print("--- Weekend rows inspection ---")
weekend_mask = df['date'].dt.dayofweek >= 5
weekend_rows = df[weekend_mask].copy()
weekend_rows['dow'] = weekend_rows['date'].dt.day_name()

print(f"  Weekend rows: {len(weekend_rows)}")
print(f"\n  NaN per weekend row:")
for _, row in weekend_rows.iterrows():
    n_nan = row[factor_cols].isna().sum()
    n_valid = len(factor_cols) - n_nan
    print(f"    {row['date'].date()} ({row['dow']}): {n_valid} valid, {n_nan} NaN out of {len(factor_cols)}")

# Which columns have values on weekends?
print(f"\n  Columns with ANY non-NaN values on weekends:")
for col in factor_cols:
    n_valid = weekend_rows[col].notna().sum()
    if n_valid > 0:
        vals = weekend_rows[col].dropna()
        print(f"    {col:<30s} {n_valid} values: {vals.tolist()[:5]}")

# Check: does the following Monday exist and have values?
print(f"\n  Does the following Monday have data?")
for _, row in weekend_rows.iterrows():
    next_monday = row['date'] + pd.Timedelta(days=(7 - row['date'].dayofweek))
    monday_row = df[df['date'] == next_monday]
    if len(monday_row) > 0:
        n_valid = monday_row.iloc[0][factor_cols].notna().sum()
        print(f"    {row['date'].date()} ({row['dow']}) → Monday {next_monday.date()}: {n_valid} valid columns")
    else:
        print(f"    {row['date'].date()} ({row['dow']}) → Monday {next_monday.date()}: NO ROW")

--- Weekend rows inspection ---
  Weekend rows: 10

  NaN per weekend row:
    2009-10-17 (Saturday): 1 valid, 45 NaN out of 46
    2012-03-03 (Saturday): 1 valid, 45 NaN out of 46
    2013-05-11 (Saturday): 1 valid, 45 NaN out of 46
    2013-11-03 (Sunday): 1 valid, 45 NaN out of 46
    2016-10-30 (Sunday): 1 valid, 45 NaN out of 46
    2019-10-27 (Sunday): 1 valid, 45 NaN out of 46
    2020-11-14 (Saturday): 1 valid, 45 NaN out of 46
    2021-07-25 (Sunday): 4 valid, 42 NaN out of 46
    2024-01-20 (Saturday): 1 valid, 45 NaN out of 46
    2024-05-18 (Saturday): 1 valid, 45 NaN out of 46

  Columns with ANY non-NaN values on weekends:
    vxd                            1 values: [12.99]
    vxdo                           1 values: [12.99]
    vxdh                           1 values: [12.99]
    vxdl                           1 values: [12.99]
    widx_ind                       9 values: [2.005104725380177e-06, 1.456712435978283e-06, -4.433774573549859e-06, 2.350894811948011e-05, -1.2

In [3]:
# %% [markdown]
# ## Stage 7: Clean & Save
#
# **Data overview:**
# Daily macro/market-level data from WRDS. 5,485 rows from 2004-01-02 to
# 2024-12-31. Keyed on date only (no PERMNO). Contains VIX family (S&P 500,
# Nasdaq-100, DJIA implied volatility OHLC), Fama-French 5 factors + momentum,
# risk-free rate, 11 FX rates, and 12 world index returns.
#
# **Columns dropped (4) — VXO discontinued:**
# - `vxo`, `vxoo`, `vxoh`, `vxol`: CBOE's old S&P 100 Volatility Index.
#   Discontinued in January 2021 — 1,041 consecutive NaN days from 2021-01-08
#   to 2024-12-31 (21.9% NaN overall). VIX is the modern replacement and has
#   full coverage.
#
# **Rows dropped (10) — weekend artefacts:**
# 6 Saturdays + 4 Sundays with at most 1 valid value out of 46 columns.
# The valid values are stray noise: `widx_ind` microreturns (~2e-06) from
# Indian market special Saturday sessions, and a single `vxd` observation.
# Every following Monday has 40–46 valid columns. Keeping these would
# pollute forward-fill (Friday → Saturday instead of Friday → Monday).
#
# **No placeholder contamination found:**
# - `fx_jpy = 99` and `fx_krw = 999` are legitimate exchange rates within
#   normal ranges (JPY: 75–162, KRW: 903–1,570).
# - >10σ outliers are real market events: `mktrf = -0.1201` (March 2020
#   COVID crash), `umd = -0.1437` (momentum crash), `widx_ind = +0.159`
#   (Indian market rally). No action needed.
#
# **`rf` (risk-free rate) retained despite only 3 unique values:**
# The daily rate was 0.0000 during ZIRP (2008–2015) and 0.0001–0.0002
# otherwise. Not useful as a standalone factor, but the merge pipeline
# needs it for computing excess returns (return - rf).
#
# **No winsorisation.** Applied in merge pipeline.
#
# **No forward-fill applied here.** NaN in remaining factors is all
# holiday-driven: VIX/FF max run 2 days (Hurricane Sandy Oct 2012),
# FX max run 2 days (Christmas), world indices max run 3–8 days
# (country-specific holidays: Chinese New Year, Japanese Golden Week).
# Forward-fill happens in the merge pipeline when aligning to trading days.
#
# **Factors retained: 42** (was 46 before cleaning)

# ═══════════════════════════════════════════════════════════════════════════════
# STAGE 7: CLEAN & SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STAGE 7: CLEAN & SAVE")
print("=" * 90)

# ── 7a. Drop VXO columns ────────────────────────────────────────────────────
drop_cols = ['vxo', 'vxoo', 'vxoh', 'vxol']
drop_cols_present = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=drop_cols_present)
print(f"\n  Dropped {len(drop_cols_present)} VXO columns (discontinued Jan 2021)")

# ── 7b. Drop weekend rows ───────────────────────────────────────────────────
n_before = len(df)
weekend_mask = df['date'].dt.dayofweek >= 5
df = df[~weekend_mask].reset_index(drop=True)
n_dropped = n_before - len(df)
print(f"  Dropped {n_dropped} weekend rows")

# ── 7c. Final NaN report ────────────────────────────────────────────────────
factor_cols_final = [c for c in df.columns if c != 'date']
nan_check = df[factor_cols_final].isna().sum()
nan_cols = nan_check[nan_check > 0].sort_values(ascending=False)
total_nan = nan_cols.sum()
total_cells = len(df) * len(factor_cols_final)

print(f"\n  Total NaN: {total_nan:,} / {total_cells:,} ({total_nan/total_cells*100:.2f}%)")
print(f"  Factors with any NaN: {len(nan_cols)} / {len(factor_cols_final)}")

print(f"\n  NaN per factor (holiday gaps, handled by forward-fill in merge):")
print(f"\n  {'Factor':<30s} {'NaN':>6s}  {'%':>6s}  {'Max Run':>8s}")
print("  " + "-" * 55)
for col in nan_cols.index:
    n = int(nan_cols[col])
    pct = n / len(df) * 100
    # Recompute max run after cleaning
    is_nan = df[col].isna()
    if is_nan.any():
        runs = is_nan.ne(is_nan.shift()).cumsum()
        nan_runs = is_nan.groupby(runs).sum()
        nan_runs = nan_runs[nan_runs > 0]
        max_run = int(nan_runs.max()) if len(nan_runs) > 0 else 0
    else:
        max_run = 0
    print(f"  {col:<30s} {n:>6,d}  {pct:>5.2f}%  {max_run:>8d}")

# ── 7d. Final summary ───────────────────────────────────────────────────────
print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Factor columns: {len(factor_cols_final)}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

# Verify no weekends remain
assert (df['date'].dt.dayofweek >= 5).sum() == 0, "Weekends still present!"
print(f"  ✓ No weekend rows")

print(f"\n  Sample (first 5 rows):")
show_cols = ['date'] + factor_cols_final[:8]
print(df[show_cols].head(5).to_string(index=False))

# ── 7e. Save ─────────────────────────────────────────────────────────────────
out_path = OUT_DIR / 'macro_daily_clean.parquet'
df.to_parquet(out_path, index=False, engine='pyarrow')
print(f"\n  ✓ Saved: {out_path}")
print(f"    {df.shape[0]:,} rows × {df.shape[1]} columns ({len(factor_cols_final)} factors)")

print("\nCleaning complete.")

STAGE 7: CLEAN & SAVE

  Dropped 4 VXO columns (discontinued Jan 2021)
  Dropped 10 weekend rows

  Total NaN: 8,977 / 229,950 (3.90%)
  Factors with any NaN: 42 / 42

  NaN per factor (holiday gaps, handled by forward-fill in merge):

  Factor                            NaN       %   Max Run
  -------------------------------------------------------
  widx_chn                          377   6.89%         8
  widx_hkg                          367   6.70%         3
  widx_jpn                          343   6.26%         6
  widx_bra                          332   6.06%         3
  widx_kor                          289   5.28%         6
  widx_ind                          287   5.24%         3
  widx_mex                          258   4.71%         4
  widx_che                          220   4.02%         3
  fx_jpy                            211   3.85%         2
  fx_gbp                            211   3.85%         2
  fx_eur                            211   3.85%         2
  fx_mxn  